# Project: YouTube Toxic Comment Detection - Phase 2
Objectives: Text Cleaning, Data Augmentation (Train only), and Multi-label Classification.

In [7]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Adding src directory to the path
sys.path.append(os.path.abspath('../'))

from src.data.loader import load_processed_data
from src.features.preprocessing import clean_text
from src.features.augmentation import augment_text # New module
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multioutput import MultiOutputClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score

## 1. Load and Preprocess Data

In [8]:
df, label_cols = load_processed_data("../data/raw/youtoxic_english_1000.csv")
df['CleanText'] = df['Text'].apply(clean_text)

# Check label distribution
print("Label Counts:")
print(df[label_cols].sum().sort_values(ascending=False))

Label Counts:
IsToxic            462
IsAbusive          353
IsProvocative      161
IsHatespeech       138
IsRacist           125
IsObscene          100
IsThreat            21
IsReligiousHate     12
IsNationalist        8
IsSexist             1
IsHomophobic         0
IsRadicalism         0
dtype: int64


## 2. Train/Test Split

In [9]:
# We use the original 'Text' for augmentation, then clean it later if needed
# But since we already have 'CleanText', let's use that.
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df[['CleanText']], df[label_cols], test_size=0.2, random_state=42
)

print(f"Original Train size: {len(X_train_raw)}")

Original Train size: 800


## 3. Data Augmentation
We augment only the training set to help the model learn minority classes.

In [10]:
# Reconstruct a temporary dataframe for the augmentation function
train_df = pd.concat([X_train_raw, y_train], axis=1).rename(columns={'CleanText': 'Text'})

# Augment minority samples (multiplier=1 means doubling the toxic samples)
augmented_train_df = augment_text(train_df, label_cols, multiplier=1)

X_train_aug = augmented_train_df['Text']
y_train_aug = augmented_train_df[label_cols]

print(f"Augmented Train size: {len(X_train_aug)}")

Starting augmentation for 355 minority samples...


100%|██████████| 355/355 [00:00<00:00, 813.26it/s]

Augmented Train size: 1155


## 4. Filtering and Vectorization

In [11]:
# Filter labels that don't have at least 2 classes in the augmented training set
valid_labels = [col for col in label_cols if y_train_aug[col].nunique() > 1]
y_train_final = y_train_aug[valid_labels]
y_test_final = y_test[valid_labels]

print(f"Active Labels for Training: {valid_labels}")

# Vectorization with tighter constraints to prevent overfitting
tfidf = TfidfVectorizer(max_features=1500, min_df=3, stop_words='english')
X_train_vec = tfidf.fit_transform(X_train_aug)
X_test_vec = tfidf.transform(X_test_raw['CleanText'])

Active Labels for Training: ['IsToxic', 'IsAbusive', 'IsThreat', 'IsProvocative', 'IsObscene', 'IsHatespeech', 'IsRacist', 'IsNationalist', 'IsSexist', 'IsReligiousHate']


## 5. Multi-label Model with Regularization

In [12]:
# C=0.1 or 0.05 for stronger regularization
model = MultiOutputClassifier(
    LogisticRegression(C=0.1, class_weight='balanced', solver='liblinear', random_state=42)
)
model.fit(X_train_vec, y_train_final)

# Predictions
y_train_pred = model.predict(X_train_vec)
y_test_pred = model.predict(X_test_vec)

# Metrics
train_f1 = f1_score(y_train_final, y_train_pred, average='micro')
test_f1 = f1_score(y_test_final, y_test_pred, average='micro')

print(f"Train F1 (Micro): {train_f1:.4f}")
print(f"Test F1 (Micro): {test_f1:.4f}")
print(f"Difference: {abs(train_f1 - test_f1)*100:.2f}%")

if abs(train_f1 - test_f1) < 0.05:
    print("✅ Level 1 Requirement Met: Overfitting is under control (< 5%).")
else:
    print("⚠️ Warning: Still overfitting. Try decreasing C or max_features.")

print("\nClassification Report (Test Set):")
print(classification_report(y_test_final, y_test_pred, target_names=valid_labels))

Train F1 (Micro): 0.8502
Test F1 (Micro): 0.5991
Difference: 25.11%
⚠️ Warning: Still overfitting. Try decreasing C or max_features.

Classification Report (Test Set):
                 precision    recall  f1-score   support

        IsToxic       0.71      0.71      0.71       107
      IsAbusive       0.69      0.73      0.71        77
       IsThreat       0.10      0.25      0.14         4
  IsProvocative       0.44      0.37      0.40        38
      IsObscene       0.36      0.57      0.44        23
   IsHatespeech       0.54      0.54      0.54        35
       IsRacist       0.53      0.63      0.58        30
  IsNationalist       0.00      0.00      0.00         3
       IsSexist       0.00      0.00      0.00         0
IsReligiousHate       0.00      0.00      0.00         4

      micro avg       0.58      0.62      0.60       321
      macro avg       0.34      0.38      0.35       321
   weighted avg       0.59      0.62      0.60       321
    samples avg       0.38      

/Users/miraekang/proyectos/ai-nlp/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/miraekang/proyectos/ai-nlp/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/miraekang/proyectos/ai-nlp/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.ca

## Conclusion

Overfitting. El modelo ha memorizado perfectamente los datos aumentados del conjunto de entrenamiento (0,85), pero no consigue rendir bien con los datos reales del conjunto de prueba (0,59). Como la diferencia es de un 25 %, sería imposible superar el nivel 1.